In [ ]:
pip install openml

In [ ]:
pip install transformers datasets torch

In [ ]:
import openml
import pandas as pd

In [ ]:
data = openml.datasets.get_dataset(1464)
data

OpenML Dataset
Name.........: blood-transfusion-service-center
Version......: 1
Format.......: ARFF
Upload Date..: 2015-05-21 22:49:48
Licence......: Public
Download URL.: https://api.openml.org/data/v1/download/1586225/blood-transfusion-service-center.arff
OpenML URL...: https://www.openml.org/d/1464
# of features: None

In [ ]:
X, y, categorical_indicator, attribute_names = data.get_data(target=data.default_target_attribute)

In [ ]:
df = pd.DataFrame(X, columns=attribute_names)
df[data.default_target_attribute] = y
df.head()

,V1,V2,V3,V4,Class
0,2,50,12500.0,98,2
1,0,13,3250.0,28,2
2,1,16,4000.0,35,2
3,2,20,5000.0,45,2
4,1,24,6000.0,77,1


- V1: Recency - months since last donation
- V2: Frequency - total number of donation
- V3: Monetary - total blood donated in c.c.
- V4: Time - months since first donation), and a binary variable representing - whether he/she donated blood in March 2007 (1 stand for donating blood; 0 stands for not donating blood).

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 748 entries, 0 to 747
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   V1      748 non-null    uint8   
 1   V2      748 non-null    uint8   
 2   V3      748 non-null    float64 
 3   V4      748 non-null    uint8   
 4   Class   748 non-null    category
dtypes: category(1), float64(1), uint8(3)
memory usage: 9.0 KB


In [ ]:
df['Class'] = df['Class'].astype(str).replace({'1': 'no', '2': 'yes'})

In [ ]:
df.Class.value_counts()

,count
Class,
no,570
yes,178


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 748 entries, 0 to 747
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   V1      748 non-null    uint8  
 1   V2      748 non-null    uint8  
 2   V3      748 non-null    float64
 3   V4      748 non-null    uint8  
 4   Class   748 non-null    object 
dtypes: float64(1), object(1), uint8(3)
memory usage: 14.0+ KB


In [ ]:
def concatenate_text_blood(row):
    text = (
        f"This person last donated blood {row['V1']} months ago. "
        f"They have donated blood {row['V2']} times in total, "
        f"with a cumulative volume of {int(row['V3'])} cc. "
        f"It has been {row['V4']} months since their first donation. "
    )
    return text

In [ ]:
# Применяем функцию к первой строке DataFrame
sample_text = concatenate_text_blood(df.iloc[0])
print(sample_text)

This person last donated blood 2 months ago. They have donated blood 50 times in total, with a cumulative volume of 12500 cc. It has been 98 months since their first donation. 


In [ ]:
df['text'] = df.apply(lambda x: concatenate_text_blood(x), axis=1)
df['text'].iloc[3]

'This person last donated blood 2 months ago. They have donated blood 20 times in total, with a cumulative volume of 5000 cc. It has been 45 months since their first donation. '

In [ ]:
from transformers import TrainingArguments, Trainer, AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
from datasets import Dataset
import torch
#import pandas as pd

In [ ]:
dataset = Dataset.from_pandas(df[['text', 'Class']])
dataset

Dataset({
    features: ['text', 'Class'],
    num_rows: 748
})

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("bigScience/T0_3B")
tokenizer = AutoTokenizer.from_pretrained("bigScience/T0_3B")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 2048)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 2048)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=2048, out_features=2048, bias=False)
              (k): Linear(in_features=2048, out_features=2048, bias=False)
              (v): Linear(in_features=2048, out_features=2048, bias=False)
              (o): Linear(in_features=2048, out_features=2048, bias=False)
              (relative_attention_bias): Embedding(32, 32)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=2048, out_features=5120, bias=False)
              (wi_1): Linear(in_features=2048, out_features=5120, bias=False)
       

In [ ]:
def tokenize_function(examples):
    inputs = tokenizer(examples['text'], padding="max_length", truncation=True, max_length=64, return_tensors="pt")
    labels = tokenizer(examples['Class'], padding="max_length", truncation=True, max_length=16, return_tensors="pt")
    inputs['labels'] = labels['input_ids']
    return inputs

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/748 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset

Dataset({
    features: ['text', 'Class', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 748
})

In [ ]:
split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

train_data = split_dataset['train']
test_data = split_dataset['test']

In [ ]:
from collections import Counter

y_values = train_data['Class']
if isinstance(y_values, list):
    print(Counter(y_values))

Counter({'no': 457, 'yes': 141})


In [ ]:
y_values_test = test_data['Class']
if isinstance(y_values_test, list):
    print(Counter(y_values_test))

Counter({'no': 113, 'yes': 37})


In [ ]:
train_data = train_data.remove_columns(['text', 'Class'])
test_data = test_data.remove_columns(['text', 'Class'])

In [ ]:
train_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

In [ ]:
print("Форма input_ids:", train_data[0]['input_ids'].shape)
print("Форма attention_mask:", train_data[0]['attention_mask'].shape)
print("Форма labels:", train_data[0]['labels'].shape)

Форма input_ids: torch.Size([64])
Форма attention_mask: torch.Size([64])
Форма labels: torch.Size([16])


In [ ]:
train_data['attention_mask'][0]

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [ ]:
train_data['input_ids'][0]

tensor([  100,   568,   336, 13207,  1717,   204,   767,   977,     5,   328,
           43, 13207,  1717,   209,   648,    16,   792,     6,    28,     3,
            9, 25965,  2908,    13,  5986,     3,    75,    75,     5,    94,
           65,   118,   204,   767,   437,    70,   166,  9294,     5,     1,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0])

In [ ]:
train_data['labels'][0]

tensor([150,   1,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0])

In [ ]:
for param in model.encoder.parameters():
    param.requires_grad = False

# Заморозка всех слоев в декодере, кроме 23-го блока
for i, layer in enumerate(model.decoder.block):
    if i != 23:  # Указание на 23-й слой (индекс 23)
        for param in layer.parameters():
            param.requires_grad = False

# Проверка заморозки
for name, param in model.named_parameters():
    print(f"{name}: requires_grad = {param.requires_grad}")

shared.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.q.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.k.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.v.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.o.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.relative_attention_bias.weight: requires_grad = False
encoder.block.0.layer.0.layer_norm.weight: requires_grad = False
encoder.block.0.layer.1.DenseReluDense.wi_0.weight: requires_grad = False
encoder.block.0.layer.1.DenseReluDense.wi_1.weight: requires_grad = False
encoder.block.0.layer.1.DenseReluDense.wo.weight: requires_grad = False
encoder.block.0.layer.1.layer_norm.weight: requires_grad = False
encoder.block.1.layer.0.SelfAttention.q.weight: requires_grad = False
encoder.block.1.layer.0.SelfAttention.k.weight: requires_grad = False
encoder.block.1.layer.0.SelfAttention.v.weight: requires_grad = False
encoder.block.1.layer.0.SelfAtt

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    save_total_limit=2,
    run_name="my_t0_experiment",
    report_to="tensorboard",  # Включить логирование
    logging_strategy="steps",  # Логировать по шагам
    logging_steps=10,         # Логировать каждые 10 шагов
    eval_strategy="steps",  # Чаще чем раз в эпоху
    eval_steps=50,            # Валидация каждые 50 шагов
    save_strategy="steps",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    disable_tqdm=False,       # Показать прогресс-бар
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
)

In [ ]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss
50,2.688000,1.786576
100,0.042100,0.037117
150,0.036800,0.032954
200,0.041900,0.032181
250,0.034600,0.032018
300,0.032000,0.031381
350,0.034200,0.031249


TrainOutput(global_step=380, training_loss=2.033815915647306, metrics={'train_runtime': 807.6314, 'train_samples_per_second': 7.404, 'train_steps_per_second': 0.471, 'total_flos': 6392856119869440.0, 'train_loss': 2.033815915647306, 'epoch': 10.0})

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score

In [ ]:
def decode_labels(labels):
    decoded_labels = []
    for label in labels:
        decoded_label = tokenizer.decode(label, skip_special_tokens=True).strip()
        decoded_labels.append(decoded_label)
    return decoded_labels

In [ ]:
def predict_tokens(input_ids):
    device = next(model.parameters()).device
    # Создаем промпт для модели
    prompt = "Did the person donate blood? Yes or no?"

    # Токенизируем промпт
    prompt_ids = tokenizer(prompt, return_tensors="pt", max_length=64, truncation=True).to(device)

    # Объединяем input_ids и prompt_ids
    input_ids = input_ids.to("cuda")
    combined_input_ids = torch.cat([input_ids, prompt_ids['input_ids']], dim=-1)

    # Генерируем ответ модели
    with torch.no_grad():
        output_ids = model.generate(input_ids=combined_input_ids, max_length=64)

    # Декодируем ответ модели в текст
    predicted_class_text = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    predicted_class = 'yes' if predicted_class_text.lower() == 'yes' else 'no'

    # Возвращаем класс, output_ids и предсказанный текст
    return predicted_class, predicted_class_text

In [ ]:
def get_predictions(test_data, sample_range=None):
    predictions = []
    true_labels = []
    predicted_texts = []

    # Определяем диапазон обработки
    if sample_range:
        start_idx, end_idx = sample_range
    else:
        start_idx, end_idx = 0, len(test_data)

    # Декодируем истинные метки
    true_labels_text = decode_labels(test_data['labels'][start_idx:end_idx])

    for i in range(start_idx, end_idx):
        example = test_data[i]

        # Получаем предсказание
        predicted_class, predicted_text = predict_tokens(example['input_ids'].unsqueeze(0))

        # Сохраняем предсказанный текст
        predicted_texts.append(predicted_text)

        # Преобразуем метки в числовой формат
        true_labels.append(1 if true_labels_text[i - start_idx] == 'yes' else 0)
        predictions.append(1 if predicted_class == 'yes' else 0)

        # Выводим предсказанный текст для каждого примера (только для выборки)
        if sample_range:
            print(f"Example {i + 1}:")
            print(f"  True label: {true_labels_text[i - start_idx]}")
            print(f"  Predicted text: {predicted_text}")
            print(f"  Predicted class: {predicted_class}")
            print("-" * 40)

    return true_labels, predictions, predicted_texts

In [ ]:
def calculate_metrics(true_labels, predictions):
    roc_auc = roc_auc_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions)
    accuracy = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions)
    recall = recall_score(true_labels, predictions)

    print(f"ROC-AUC: {roc_auc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")

In [ ]:
true_labels_full, predictions_full, _ = get_predictions(test_data)

# Вычисляем метрики для полного набора
calculate_metrics(true_labels_full, predictions_full)

ROC-AUC: 0.6135
F1 Score: 0.4000
Accuracy: 0.7600
Precision: 0.5217
Recall: 0.3243


In [ ]:
# Пример использования
start_idx = 0  # Начальный индекс выборки
end_idx = 149   # Конечный индекс выборки
true_labels_sample, predictions_sample, _ = get_predictions(test_data, sample_range=(start_idx, end_idx))

Example 1:
  True label: no
  Predicted text: no
  Predicted class: no
----------------------------------------
Example 2:
  True label: no
  Predicted text: no
  Predicted class: no
----------------------------------------
Example 3:
  True label: no
  Predicted text: no
  Predicted class: no
----------------------------------------
Example 4:
  True label: yes
  Predicted text: yes
  Predicted class: yes
----------------------------------------
Example 5:
  True label: no
  Predicted text: yes
  Predicted class: yes
----------------------------------------
Example 6:
  True label: no
  Predicted text: yes
  Predicted class: yes
----------------------------------------
Example 7:
  True label: no
  Predicted text: yes
  Predicted class: yes
----------------------------------------
Example 8:
  True label: no
  Predicted text: no
  Predicted class: no
----------------------------------------
Example 9:
  True label: no
  Predicted text: no
  Predicted class: no
------------------------

In [ ]:
prediction_counts = Counter(predictions_full)

print(prediction_counts)

Counter({0: 127, 1: 23})
